# Data Science Fundamentals with Python

This notebook covers essential data science concepts using Python libraries.

## Learning Objectives:
- Load and explore datasets
- Perform data cleaning and preprocessing
- Create visualizations
- Apply statistical analysis


In [ ]:
# Import essential libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set style for better plots
plt.style.use('default')
sns.set_palette('husl')

print("Libraries imported successfully!")

## 1. Data Loading and Exploration

In [ ]:
# Create sample dataset
np.random.seed(42)
n_samples = 1000

data = {
    'age': np.random.normal(35, 10, n_samples),
    'income': np.random.normal(50000, 15000, n_samples),
    'education_years': np.random.normal(16, 3, n_samples),
    'experience': np.random.normal(10, 5, n_samples),
    'satisfaction': np.random.normal(7, 2, n_samples)
}

# Apply constraints
data['age'] = np.clip(data['age'], 18, 65)
data['income'] = np.clip(data['income'], 20000, 150000)
data['education_years'] = np.clip(data['education_years'], 12, 25)
data['experience'] = np.clip(data['experience'], 0, 40)
data['satisfaction'] = np.clip(data['satisfaction'], 1, 10)

df = pd.DataFrame(data)
print(f"Dataset created with shape: {df.shape}")
df.head()

In [ ]:
# Basic information about the dataset
print("Dataset Info:")
print(df.info())
print("\nDescriptive Statistics:")
print(df.describe())

## 2. Data Visualization

In [ ]:
# Create comprehensive visualization
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Age distribution
axes[0, 0].hist(df['age'], bins=30, alpha=0.7, color='skyblue', edgecolor='black')
axes[0, 0].set_title('Age Distribution', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Age')
axes[0, 0].set_ylabel('Frequency')

# Income vs Experience scatter plot
axes[0, 1].scatter(df['experience'], df['income'], alpha=0.6, color='green')
axes[0, 1].set_title('Income vs Experience', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Experience (years)')
axes[0, 1].set_ylabel('Income ($)')

# Satisfaction by education level
education_bins = pd.cut(df['education_years'], bins=3, labels=['Low', 'Medium', 'High'])
df_temp = df.copy()
df_temp['education_level'] = education_bins
df_temp.boxplot(column='satisfaction', by='education_level', ax=axes[0, 2])
axes[0, 2].set_title('Satisfaction by Education Level', fontsize=14, fontweight='bold')
axes[0, 2].set_xlabel('Education Level')

# Correlation heatmap
correlation = df.corr()
im = axes[1, 0].imshow(correlation, cmap='coolwarm', aspect='auto')
axes[1, 0].set_xticks(range(len(correlation.columns)))
axes[1, 0].set_yticks(range(len(correlation.columns)))
axes[1, 0].set_xticklabels(correlation.columns, rotation=45)
axes[1, 0].set_yticklabels(correlation.columns)
axes[1, 0].set_title('Correlation Heatmap', fontsize=14, fontweight='bold')

# Add correlation values
for i in range(len(correlation.columns)):
    for j in range(len(correlation.columns)):
        text = axes[1, 0].text(j, i, f'{correlation.iloc[i, j]:.2f}',
                              ha="center", va="center", color="black", fontweight='bold')

# Income distribution
axes[1, 1].hist(df['income'], bins=30, alpha=0.7, color='orange', edgecolor='black')
axes[1, 1].set_title('Income Distribution', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Income ($)')
axes[1, 1].set_ylabel('Frequency')

# Age vs Satisfaction scatter plot
axes[1, 2].scatter(df['age'], df['satisfaction'], alpha=0.6, color='red')
axes[1, 2].set_title('Age vs Satisfaction', fontsize=14, fontweight='bold')
axes[1, 2].set_xlabel('Age')
axes[1, 2].set_ylabel('Satisfaction Score')

plt.tight_layout()
plt.show()

## 3. Statistical Analysis

In [ ]:
# Correlation analysis
print("=== Correlation Analysis ===")
income_experience_corr = df['income'].corr(df['experience'])
print(f"Income-Experience correlation: {income_experience_corr:.3f}")

age_satisfaction_corr = df['age'].corr(df['satisfaction'])
print(f"Age-Satisfaction correlation: {age_satisfaction_corr:.3f}")

# Statistical significance test
print("\n=== Hypothesis Testing ===")
high_education = df[df['education_years'] > df['education_years'].median()]
low_education = df[df['education_years'] <= df['education_years'].median()]

# T-test
t_stat, p_value = stats.ttest_ind(high_education['satisfaction'], low_education['satisfaction'])
print(f"T-test for satisfaction difference by education:")
print(f"t-statistic: {t_stat:.3f}")
print(f"p-value: {p_value:.3f}")

if p_value < 0.05:
    print("Result: Significant difference in satisfaction between education levels")
else:
    print("Result: No significant difference in satisfaction between education levels")

## 4. Data Preprocessing

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Feature scaling demonstration
print("=== Feature Scaling ===")
print("Original data statistics:")
print(df.describe())

# Standardization (Z-score normalization)
scaler_standard = StandardScaler()
df_standardized = pd.DataFrame(
    scaler_standard.fit_transform(df),
    columns=df.columns
)

print("\nStandardized data statistics:")
print(df_standardized.describe())

# Min-Max scaling
scaler_minmax = MinMaxScaler()
df_minmax = pd.DataFrame(
    scaler_minmax.fit_transform(df),
    columns=df.columns
)

print("\nMin-Max scaled data statistics:")
print(df_minmax.describe())

## 5. Outlier Detection

In [ ]:
# Outlier detection using IQR method
def detect_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    return outliers, lower_bound, upper_bound

print("=== Outlier Detection (IQR Method) ===")
for column in df.columns:
    outliers, lower, upper = detect_outliers_iqr(df, column)
    print(f"{column}: {len(outliers)} outliers (bounds: {lower:.2f} - {upper:.2f})")

# Visualize outliers for income
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Box plot
ax1.boxplot(df['income'])
ax1.set_title('Income Box Plot (Outlier Detection)')
ax1.set_ylabel('Income ($)')

# Histogram with outlier bounds
outliers_income, lower_income, upper_income = detect_outliers_iqr(df, 'income')
ax2.hist(df['income'], bins=30, alpha=0.7, edgecolor='black')
ax2.axvline(lower_income, color='red', linestyle='--', label=f'Lower bound: {lower_income:.0f}')
ax2.axvline(upper_income, color='red', linestyle='--', label=f'Upper bound: {upper_income:.0f}')
ax2.set_title('Income Distribution with Outlier Bounds')
ax2.set_xlabel('Income ($)')
ax2.set_ylabel('Frequency')
ax2.legend()

plt.tight_layout()
plt.show()

## Summary and Next Steps

In this notebook, we've covered:

1. **Data Loading**: Creating and loading datasets
2. **Exploratory Data Analysis**: Understanding data structure and distributions
3. **Data Visualization**: Creating meaningful plots and charts
4. **Statistical Analysis**: Correlation analysis and hypothesis testing
5. **Data Preprocessing**: Feature scaling and normalization
6. **Outlier Detection**: Identifying and visualizing outliers

### Next Steps:
- Practice with real-world datasets
- Learn advanced visualization techniques
- Explore machine learning algorithms
- Build predictive models
